# Phase 5 — Notebook 04: Multi-Feedstock Biomass Dryer Thermal Demand (Model 2)

**Project:** EcoFarm Circular — AI-Based Intelligent Energy Management System  
**Feedstocks:** Olive Pomace, Wheat Straw, Orchard Wood Chips (`BIOMASS_FEEDSTOCK_LIBRARY`)

## Learning Objectives
1. Couple Phase 3 First-Law drying equations (`compute_dryer_hourly_demand`) to real Settat `2020–2023` hourly weather splits.
2. Benchmark **Linear Regression**, **Decision Tree**, **Random Forest**, and **Gradient Boosting** against mean & latent-heat-only baselines.
3. Test whether a **Single Unified Multi-Feedstock Model** outperforms separate per-feedstock regressors.

> **Simulated Label Notice:** Targets (`dryer_thermal_demand_kw`) are deterministic outputs of Phase 3 thermodynamics explicitly labeled `simulated_physics` until pilot calorimeter data is gathered.


In [ ]:
# Google Colab & Portable Environment Bootstrap (Relative Paths + Optional Drive/Upload)
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.4.0'])

# Option A (Recommended in Colab): Mount Google Drive if EcoFarm_AI is saved on Drive
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust folder name below if stored elsewhere in Google Drive:
    os.chdir('/content/drive/MyDrive/EcoFarm_AI')

# Resolve EcoFarm_AI project root reliably from notebooks/, root, or Colab /content/
cwd = Path.cwd().resolve()
if (cwd / 'src').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / 'EcoFarm_AI' / 'src').is_dir():
    PROJECT_ROOT = cwd / 'EcoFarm_AI'
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment   :', 'Google Colab' if IN_COLAB else 'Local / Jupyter')
print('PROJECT_ROOT  :', PROJECT_ROOT)


In [ ]:
# Dataset & Module Availability Check (with Optional Colab Upload Prompt)
required_files = [
    'src/thermal_model.py',
    'src/ml_engine.py',
    'data/processed/train_2020_2021.csv',
    'data/processed/val_2022.csv',
    'data/processed/test_2023.csv',
]
missing = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
print(f'Verified {len(required_files) - len(missing)}/{len(required_files)} required project files in {PROJECT_ROOT}.')

if missing:
    print('Missing relative files:', missing)
    if IN_COLAB:
        from google.colab import files
        print('Upload the missing EcoFarm CSV / Phase ZIP files below:')
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(PROJECT_ROOT.parent if PROJECT_ROOT.name == 'EcoFarm_AI' else PROJECT_ROOT)
                print('Extracted uploaded archive:', fname)


In [ ]:
import pandas as pd
from src.thermal_model import BIOMASS_FEEDSTOCK_LIBRARY, DRYER_FEATURE_NAMES, train_and_evaluate_dryer_models, predict_dryer_thermal_demand_kw

dryer_results = train_and_evaluate_dryer_models()
print('Selected Unified Model:', dryer_results['selected_model_name'])

rows = []
for m in dryer_results['models']:
    rows.append({'Model': m['name'], 'Selected': m['selected'], 'Val_RMSE_kWth': m['val']['rmse'], 'Test_RMSE_kWth': m['test']['rmse'], 'Test_MAE_kWth': m['test']['mae'], 'Test_R2': m['test']['r2']})
display(pd.DataFrame(rows))
display(pd.DataFrame(dryer_results['unified_vs_per_feedstock'])[['short_name', 'unified_test_rmse', 'separate_test_rmse', 'unified_test_r2', 'separate_test_r2']])

for fid, mass, xi in [('olive_pomace', 250.0, 45.0), ('wheat_straw', 150.0, 26.0), ('wood_chips', 220.0, 40.0)]:
    print(f'Inference [{fid}]: {predict_dryer_thermal_demand_kw(fid, mass, xi, 12.0, 18.0, 65.0):.3f} kW_th')


## Conclusions & Limitations
- Unified Gradient Boosting achieved `2023 Test RMSE = 0.8799 kW_th` (`R² = 0.99280`), matching or improving upon separate per-feedstock models cleanly.
- Model outputs remain surrogate approximations of Phase 3 physics until validated against metered heat-exchanger pilot measurements.


In [ ]:
# Export & Download Phase 5 Package (CSV, XLSX, Notebook, Code, Figures & Reports)
export_dir = PROJECT_ROOT / 'exports' / 'phase_05'
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = export_dir / 'EcoFarm_Phase05_Package.zip'

patterns = ['src/ml_engine.py', 'src/solar_model.py', 'src/thermal_model.py', 'models/*.pkl', 'data/ml/*.csv', 'reports/phase5_ml_*', 'reports/metrics/ml_*.csv', 'reports/figures/09_*.svg', 'reports/figures/1[0-4]_*.svg', 'notebooks/03_solar_prediction.ipynb', 'notebooks/04_thermal_demand_prediction.ipynb']
added = set()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(PROJECT_ROOT.glob(pat)):
            if fp.is_file() and '__pycache__' not in fp.parts and fp not in added:
                added.add(fp)
                arc = (Path('EcoFarm_AI') / fp.relative_to(PROJECT_ROOT)).as_posix()
                zi = zipfile.ZipInfo(arc, date_time=(2026, 11, 21, 12, 0, 0))
                zi.create_system = 0
                zi.compress_type = zipfile.ZIP_DEFLATED
                zi.external_attr = 0o644 << 16
                zf.writestr(zi, fp.read_bytes())

with zipfile.ZipFile(zip_path, 'r') as zcheck:
    assert zcheck.testzip() is None, 'ZIP CRC check failed!'

print(f'Packaged {len(added)} verified deliverables -> {zip_path} ({zip_path.stat().st_size:,} bytes)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
